<a href="https://colab.research.google.com/github/JariusF10-8/bt4222/blob/main/notebooks/features/feature_engineering_zp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
from pathlib import Path
import io
import os
import sys
import zipfile
import tempfile
import subprocess
import importlib

# Upload the original repository ZIP from Downloads.
uploaded = files.upload()
zip_name = next(
    name for name in uploaded
    if name.lower().endswith('.zip')
)

# Extract into a fresh folder.
extract_dir = Path(tempfile.mkdtemp(
    prefix='bt4222_', dir='/content'
))

with zipfile.ZipFile(io.BytesIO(uploaded[zip_name])) as archive:
    archive.extractall(extract_dir)

# Locate the actual folder containing src/preprocessing.py.
matches = list(extract_dir.rglob('src/preprocessing.py'))

if len(matches) != 1:
    raise FileNotFoundError(
        f'Expected one src/preprocessing.py; found {len(matches)}. '
        'Check that you uploaded the preprocessing branch ZIP.'
    )

project_root = matches[0].parent.parent

os.environ['BT4222_PROJECT_ROOT'] = str(project_root)
os.environ['MIND_RAW_DIR'] = str(project_root / 'data/raw')

subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q',
     '-r', str(project_root / 'requirements.txt')],
    check=True,
)

sys.path.insert(0, str(project_root))
importlib.invalidate_caches()

from src.preprocessing import build_dataset

print('Import successful!')
print('Actual project folder:', project_root)

Saving bt4222-main.zip to bt4222-main.zip
Import successful!
Actual project folder: /content/bt4222_cwor_nu3/bt4222-main


In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

# Keep the project/raw-data paths set by the successful setup cell.
print('Project:', os.environ['BT4222_PROJECT_ROOT'])
print('Raw data:', os.environ['MIND_RAW_DIR'])

# Save preprocessing outputs in Drive.
os.environ['BT4222_PROCESSED_DIR'] = (
    '/content/drive/MyDrive/BT4222 Group Project/feature_engineering_zp/data/processed/v2'
)

# Save engineered features in Drive.
os.environ['BT4222_NEWS_FEATURES_DIR'] = (
    '/content/drive/MyDrive/BT4222 Group Project/feature_engineering_zp/outputs/features/news_time_v3'
)

Mounted at /content/drive
Project: /content/bt4222_cwor_nu3/bt4222-main
Raw data: /content/bt4222_cwor_nu3/bt4222-main/data/raw


In [ ]:
from IPython.display import display

raw_dir = Path(os.environ['MIND_RAW_DIR'])
processed_dir = Path(os.environ['BT4222_PROCESSED_DIR'])

if (processed_dir / 'manifest.json').is_file():
    print('Using existing preprocessed data:', processed_dir)
else:
    split_summary = build_dataset(
        raw_dir=raw_dir,
        output_dir=processed_dir,
        require_test=True,
        overwrite=False,
    )
    display(split_summary)
    print('Created preprocessed data:', processed_dir)

Preparing train: 95,071 events
Preparing validation: 61,894 events
Preparing test: 73,152 events


,partition,earliest_time,latest_time,events,users,candidate_exposures,clicked_candidates,nonclicked_candidates,events_with_empty_history,referenced_articles,articles_missing_metadata,candidate_exposures_missing_metadata
0,train,2019-11-09 00:00:19,2019-11-12 23:59:25,95071,40148,3389342,141558,3247784,1735,43370,0,0
1,validation,2019-11-13 00:01:00,2019-11-14 23:59:13,61894,31664,2454102,94786,2359316,1503,38421,0,0
2,test,2019-11-15 00:00:01,2019-11-15 23:58:03,73152,50000,2740998,111383,2629615,2214,42416,0,0


Created preprocessed data: /content/drive/MyDrive/BT4222 Group Project/feature_engineering_zp/data/processed/v2


# MIND-small: basic news features and impression-time context

Start from the CSVs produced by `notebooks/01_Data_Preprocessing.ipynb` on the `preprocessing` branch. This update preserves the three news features and adds time context plus candidate-category and historical-interest interactions.

- **News:** `title_length`, `abstract_length`, `num_title_entities`. Recompute and verify the values already supplied by preprocessing.
- **Impressions:** derive time-of-day, day-of-week and weekend features from `events.csv` (`timestamp`, originally `behaviors.tsv`'s impression time).
- **Interactions:** combine time with candidate category and the category interests in the impression's supplied prior history. Do not hard-code assumptions such as entertainment on weekends.

**Run:** save in `notebooks/features/`, run preprocessing first, configure paths below, then run top to bottom. Dependencies: Python, pandas and NumPy. No text model, embeddings or click labels are used in these calculations.

The timestamp describes the current recommendation context, not an article's publication time or a permanent user attribute. Source times have no documented timezone here; keep them as recorded rather than interpreting them as each user's local clock.


## 1. Feature definitions

| Feature | Original source | Calculation | Data type / unit |
|---|---|---|---|
| `title_length` | `news.tsv` → title | Number of whitespace-separated words, using `str.split()` | int32 / words |
| `abstract_length` | `news.tsv` → abstract | Number of whitespace-separated words, using `str.split()` | int32 / words |
| `num_title_entities` | `news.tsv` → title entity JSON | Number of annotation objects in the parsed list | int32 / entities |

Punctuation is retained: `Apple's new iPhone!` has 3 words. Multiple spaces, tabs and newlines are treated as whitespace. Blank/missing text and empty entity lists yield 0. Literal text such as `NA` stays text.

Entity count means **annotation entries**, not number of mentions, occurrence offsets, surface forms, or deduplicated Wikidata IDs. One annotation with several occurrence offsets still counts as one. No confidence threshold is applied.

The output also retains `metadata_missing`, `title_missing`, `abstract_missing` and `title_entities_missing` so a zero due to missing information can be distinguished from an observed zero.


## 2. Configure paths

Defaults follow the repository layout. To use a different location, edit the paths below or set `BT4222_PROJECT_ROOT`, `BT4222_PROCESSED_DIR` and/or `BT4222_NEWS_FEATURES_DIR` before running.

In Colab, clone/open the repository first. If your processed files are in Drive, mount Drive and point `PROCESSED_DIR` at the **whole version folder** containing `manifest.json` and the partition folders. This notebook does not rerun preprocessing or download raw data. The default output folder is `outputs/features/news_time_v2`; the existing `BT4222_NEWS_FEATURES_DIR` override is still supported.


In [ ]:
from pathlib import Path
import hashlib
import json
import os

import pandas as pd
import numpy as np
import warnings
try:
    from IPython.display import display
except ImportError:  # Also permit running the code as a plain Python script.
    display = print

# Optional Colab setup, when the processed version is on Google Drive:
# from google.colab import drive
# drive.mount('/content/drive')

configured_root = os.environ.get('BT4222_PROJECT_ROOT')
PROJECT_ROOT = (
    Path(configured_root).expanduser().resolve()
    if configured_root else next(
        (p for p in [Path.cwd(), *Path.cwd().parents]
         if (p / 'src/data_loading.py').is_file()), None
    )
)

# Edit these paths if needed, e.g. PROCESSED_DIR = Path('/content/drive/MyDrive/.../v1').
configured_processed = os.environ.get('BT4222_PROCESSED_DIR')
if configured_processed:
    PROCESSED_DIR = Path(configured_processed).expanduser().resolve()
elif PROJECT_ROOT is not None:
    PROCESSED_DIR = PROJECT_ROOT / 'data/processed/v1'
else:
    raise FileNotFoundError(
        'Open this notebook inside the cloned repository, or set '
        'BT4222_PROCESSED_DIR to the preprocessed version folder.'
    )

default_output = (
    PROJECT_ROOT / 'outputs/features/news_time_v2'
    if PROJECT_ROOT is not None
    else PROCESSED_DIR.parent / 'news_time_features_v2'
)
OUTPUT_DIR = Path(os.environ.get('BT4222_NEWS_FEATURES_DIR', str(default_output))).expanduser().resolve()

if OUTPUT_DIR == PROCESSED_DIR or PROCESSED_DIR in OUTPUT_DIR.parents:
    raise ValueError('Choose an OUTPUT_DIR outside the preprocessed version folder.')
print('Preprocessed input:', PROCESSED_DIR)
print('Feature output:    ', OUTPUT_DIR)


Preprocessed input: /content/drive/MyDrive/BT4222 Group Project/feature_engineering_zp/data/processed/v2
Feature output:     /content/drive/MyDrive/BT4222 Group Project/feature_engineering_zp/outputs/features/news_time_v3


## 3. Read available partitions

Use the preprocessing manifest to discover `train`, `validation` and, if available, `test`. Preserve the existing chronological split. A train/validation-only version is supported; no test partition is invented.

This first section loads `news.csv`. Later sections load the compact event table and read candidate rows only in small batches for interactions. CSVs are read as strings with `keep_default_na=False`, matching the shared loader's text handling; JSON lists and missingness flags are explicitly restored below.


In [ ]:
manifest_path = PROCESSED_DIR / 'manifest.json'
if not manifest_path.is_file():
    raise FileNotFoundError(
        f'{manifest_path} is missing. Run 01_Data_Preprocessing.ipynb first '
        'or correct PROCESSED_DIR.'
    )
source_manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
PARTITIONS = source_manifest.get('partitions')
if (not isinstance(PARTITIONS, list) or not PARTITIONS
        or any(p not in ['train', 'validation', 'test'] for p in PARTITIONS)
        or len(PARTITIONS) != len(set(PARTITIONS))):
    raise ValueError('Expected a nonempty list of unique train/validation/test partitions.')
for partition in PARTITIONS:
    path = PROCESSED_DIR / partition / 'news.csv'
    if not path.is_file():
        raise FileNotFoundError(f'Manifest lists {partition!r}, but {path} is missing.')

print('Partitions:', ', '.join(PARTITIONS))
if 'test' not in PARTITIONS:
    print('No test partition is available in this preprocessed version.')


Partitions: train, validation, test


## 4. Derive and verify features

The functions below preserve one row per `news_id`. Malformed entity JSON and inconsistent existing counts raise clear errors instead of silently changing the data.

These are deterministic, per-article calculations. Apply the same definitions independently to each partition; no fitting or pooling across partitions is required. Any later scaler or learned transformation must be fitted on training data only.


In [ ]:
FEATURE_COLUMNS = ['title_length', 'abstract_length', 'num_title_entities']
FLAG_COLUMNS = ['metadata_missing', 'title_missing', 'abstract_missing', 'title_entities_missing']


def parse_title_entities(value, news_id):
    # The canonical export stores parsed lists as JSON strings.
    try:
        parsed = json.loads(value)
    except (TypeError, json.JSONDecodeError) as exc:
        raise ValueError(f'{news_id}: invalid title_entities_parsed JSON') from exc
    if not isinstance(parsed, list) or not all(isinstance(e, dict) for e in parsed):
        raise ValueError(f'{news_id}: title_entities_parsed must be a list of objects')
    return parsed


def derive_news_features(news):
    required = ['news_id', 'title', 'abstract', 'title_entities_parsed', *FLAG_COLUMNS]
    missing = sorted(set(required) - set(news.columns))
    if missing:
        raise ValueError(f'Preprocessed news table is missing columns: {missing}')
    if news.empty:
        raise ValueError('Preprocessed news table has no rows.')
    if not news['news_id'].str.fullmatch(r'N[0-9]+', na=False).all():
        raise ValueError('Missing or invalid news_id in preprocessed news table.')
    if not news['news_id'].is_unique:
        raise ValueError('Duplicate news_id would multiply downstream joins.')

    entities = pd.Series(
        [parse_title_entities(value, news_id)
         for news_id, value in zip(news['news_id'], news['title_entities_parsed'])],
        index=news.index, dtype='object',
    )
    features = news[['news_id']].copy()
    features['title_length'] = news['title'].fillna('').str.split().str.len().astype('int32')
    features['abstract_length'] = news['abstract'].fillna('').str.split().str.len().astype('int32')
    features['num_title_entities'] = entities.map(len).astype('int32')

    # Current preprocessing already exports all three counts; verify rather than duplicate.
    checked = []
    for column in FEATURE_COLUMNS:
        if column in news.columns:
            existing = pd.to_numeric(news[column], errors='raise')
            matches = existing.eq(features[column])
            if not matches.all():
                ids = news.loc[~matches, 'news_id'].head(5).tolist()
                raise ValueError(f'{column} disagrees with preprocessing for news IDs: {ids}')
            checked.append(column)

    for column in FLAG_COLUMNS:
        if not news[column].isin(['True', 'False']).all():
            raise ValueError(f'{column} must contain True/False in the canonical CSV.')
        features[column] = news[column].eq('True').astype('bool')

    if features[FEATURE_COLUMNS].isna().any().any() or (features[FEATURE_COLUMNS] < 0).any().any():
        raise ValueError('Derived counts must be non-missing, nonnegative integers.')
    if not features['news_id'].equals(news['news_id']):
        raise ValueError('Feature derivation changed article IDs or their order.')
    return features, checked


In [ ]:
features_by_partition = {}
summary_rows = []

for partition in PARTITIONS:
    news = pd.read_csv(
        PROCESSED_DIR / partition / 'news.csv',
        dtype='string', keep_default_na=False, encoding='utf-8',
    )
    features, checked = derive_news_features(news)
    features_by_partition[partition] = features
    summary_rows.append({
        'partition': partition,
        'articles': len(features),
        'existing_feature_columns_verified': ', '.join(checked) or '(none present)',
        'metadata_missing': int(features['metadata_missing'].sum()),
        'mean_title_length': features['title_length'].mean(),
        'mean_abstract_length': features['abstract_length'].mean(),
        'mean_num_title_entities': features['num_title_entities'].mean(),
    })
    print(f'{partition}: {len(features):,} articles; verified {len(checked)} existing feature columns')

summary = pd.DataFrame(summary_rows)
display(summary)
preview_partition = 'train' if 'train' in features_by_partition else PARTITIONS[0]
display(features_by_partition[preview_partition].head(10))


train: 43,370 articles; verified 3 existing feature columns
validation: 38,421 articles; verified 3 existing feature columns
test: 42,416 articles; verified 3 existing feature columns


,partition,articles,existing_feature_columns_verified,metadata_missing,mean_title_length,mean_abstract_length,mean_num_title_entities
0,train,43370,"title_length, abstract_length, num_title_entities",0,10.759765,33.757690,1.170602
1,validation,38421,"title_length, abstract_length, num_title_entities",0,10.808386,33.047422,1.158741
2,test,42416,"title_length, abstract_length, num_title_entities",0,10.804649,33.103475,1.176773


,news_id,title_length,abstract_length,num_title_entities,metadata_missing,title_missing,abstract_missing,title_entities_missing
0,N10,6,5,1,False,False,False,False
1,N100,11,68,1,False,False,False,False
2,N1000,9,30,2,False,False,False,False
3,N10000,8,8,0,False,False,False,False
4,N10001,14,24,2,False,False,False,False
5,N10002,12,138,1,False,False,False,False
6,N10003,10,43,0,False,False,False,False
7,N10004,7,10,3,False,False,False,False
8,N10005,10,15,2,False,False,False,False
9,N10009,14,30,2,False,False,False,False


## 5. Export the basic news feature tables

Each partition receives `news_features.csv`, with `news_id`, the three derived features and four missingness flags. Existing preprocessing files remain the inputs. Running this cell again refreshes only this notebook's named output files.

`feature_dictionary.csv`, `summary.csv` and a small provenance manifest accompany the feature tables. An article may appear in more than one partition; keep its partition-specific metadata and feature table together.


In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
for partition, features in features_by_partition.items():
    folder = OUTPUT_DIR / partition
    folder.mkdir(parents=True, exist_ok=True)
    features.to_csv(folder / 'news_features.csv', index=False, encoding='utf-8')

feature_dictionary = pd.DataFrame([
    {'feature': 'title_length', 'source': 'News',
     'description': 'Number of whitespace-separated words in title',
     'data_type': 'int32', 'unit': 'Word count', 'status': 'Derived'},
    {'feature': 'abstract_length', 'source': 'News',
     'description': 'Number of whitespace-separated words in abstract',
     'data_type': 'int32', 'unit': 'Word count', 'status': 'Derived'},
    {'feature': 'num_title_entities', 'source': 'News',
     'description': 'Number of title entity annotation objects (not mentions)',
     'data_type': 'int32', 'unit': 'Count', 'status': 'Derived'},
])
feature_dictionary.to_csv(OUTPUT_DIR / 'feature_dictionary.csv', index=False, encoding='utf-8')
summary.to_csv(OUTPUT_DIR / 'summary.csv', index=False, encoding='utf-8')


def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()


provenance = {
    'source_processed_directory': str(PROCESSED_DIR),
    'source_dataset_version': source_manifest.get('dataset_version'),
    'source_manifest_sha256': sha256_file(manifest_path),
    'source_news_sha256': {
        p: sha256_file(PROCESSED_DIR / p / 'news.csv') for p in PARTITIONS
    },
    'partitions': PARTITIONS,
    'feature_columns': FEATURE_COLUMNS,
    'missingness_flags': FLAG_COLUMNS,
    'count_dtype_on_load': 'int32 (CSV does not retain pandas dtypes)',
    'text_length_definition': 'len(text.split()); blank/missing text gives 0',
    'entity_count_definition': 'number of objects in title_entities_parsed',
    'reference_branch_tree': '66c07074e9c720e8f84e17a8172b1542d34b968c',
    'pandas_version': pd.__version__,
}
(OUTPUT_DIR / 'manifest.json').write_text(
    json.dumps(provenance, indent=2) + '\n', encoding='utf-8'
)
print('Saved feature tables and documentation to:', OUTPUT_DIR)


Saved feature tables and documentation to: /content/drive/MyDrive/BT4222 Group Project/feature_engineering_zp/outputs/features/news_time_v3


## 6. Implementing a time based feature based on Professor's suggestions

Use time as a feature to potentially distinguish context. The data must establish whether it helps; weekday/business and weekend/entertainment are examples to test, not rules to assign.

Models can already read an hour represented as a number. **Cyclic encoding preserves the relationship between times:** 23:00 and 00:00 are one hour apart, although the raw numbers 23 and 0 are far apart. Treat time as a point on a 24-hour clock:

$$
\begin{aligned}
h &= \text{hour} + \frac{\text{minute}}{60}
     + \frac{\text{second}}{3600} \\
\text{hour\_sin} &= \sin\left(\frac{2\pi h}{24}\right) \\
\text{hour\_cos} &= \cos\left(\frac{2\pi h}{24}\right)
\end{aligned}
$$

Use **both** values: sine alone gives the same value at midnight and noon, while the pair distinguishes them. Both are bounded by -1 and 1 and require no fitted scaler. A compact pair is a starting choice, not a guarantee of optimal performance; hour bins or richer periodic features are possible later ablations.

The weekday pair uses the same formula with period 7 and Monday=0 through Sunday=6. `is_weekend` is 1 for recorded Saturday/Sunday, otherwise 0.

**Dataset limitation:** the repository's split is train **9–12 November 2019 (Saturday–Tuesday)**, validation **13–14 November (Wednesday–Thursday)**, and optional later dev events as test. Training contains no repeated weekly cycle, and held-out weekdays differ. We derive all the requested features, but the main configuration uses hour features only. Set `USE_WEEKDAY_FEATURES=True` for an explicitly labelled experiment. Weekend/weekday results in this short window can reflect particular dates and news events, not a stable habit. Keep the chronological split.


In [ ]:
# Decide this before model selection. All weekday fields are exported either way.
USE_WEEKDAY_FEATURES = False

# Candidate-level tables can be large; the batch builder below always works.
# Set True only if you also want all candidate interactions written as CSV.
EXPORT_CANDIDATE_CONTEXT = False
CANDIDATE_CHUNK_SIZE = 100_000

ALL_TIME_FEATURE_COLUMNS = ['hour_sin', 'hour_cos', 'weekday_sin', 'weekday_cos', 'is_weekend']
TIME_FEATURE_COLUMNS = ['hour_sin', 'hour_cos'] + (
    ['weekday_sin', 'weekday_cos', 'is_weekend'] if USE_WEEKDAY_FEATURES else []
)


def derive_event_time_features(events):
    required = ['event_key', 'timestamp']
    missing = sorted(set(required) - set(events.columns))
    if missing:
        raise ValueError(f'Event table is missing columns: {missing}')
    keys = events['event_key']
    if events.empty or not keys.fillna('').str.strip().ne('').all() or not keys.is_unique:
        raise ValueError('Expected nonempty events with unique, nonblank event_key values.')
    timestamp = pd.to_datetime(events['timestamp'], format='%Y-%m-%d %H:%M:%S', errors='raise')
    if timestamp.isna().any():
        raise ValueError('Missing impression timestamp; no time imputation is performed.')
    if timestamp.dt.tz is not None:
        raise ValueError('Expected canonical timezone-naive timestamps; do not silently convert zones.')

    fractional_hour = (
        timestamp.dt.hour + timestamp.dt.minute / 60.0 + timestamp.dt.second / 3600.0
    )
    weekday = timestamp.dt.dayofweek
    out = events[['event_key']].copy()
    out['timestamp'] = timestamp  # Audit field, excluded from model input columns.
    out['hour_of_day'] = timestamp.dt.hour.astype('int8')
    out['fractional_hour'] = fractional_hour.astype('float32')
    out['day_of_week'] = weekday.astype('int8')  # Monday=0, Sunday=6.
    out['is_weekend'] = weekday.ge(5).astype('int8')
    # Compute phases before rounding to float32; keep minutes and seconds.
    out['hour_sin'] = np.sin(2 * np.pi * fractional_hour / 24).astype('float32')
    out['hour_cos'] = np.cos(2 * np.pi * fractional_hour / 24).astype('float32')
    out['weekday_sin'] = np.sin(2 * np.pi * weekday / 7).astype('float32')
    out['weekday_cos'] = np.cos(2 * np.pi * weekday / 7).astype('float32')
    if not np.isfinite(out[ALL_TIME_FEATURE_COLUMNS].to_numpy()).all():
        raise ValueError('Time features contain non-finite values.')
    return out


demo_hours = np.array([0, 6, 12, 18, 23], dtype='float64')
display(pd.DataFrame({
    'hour': demo_hours.astype(int),
    'hour_sin': np.sin(2 * np.pi * demo_hours / 24),
    'hour_cos': np.cos(2 * np.pi * demo_hours / 24),
}).round(3))
print('Selected time model inputs:', TIME_FEATURE_COLUMNS)


,hour,hour_sin,hour_cos
0,0,0.000,1.000
1,6,1.000,0.000
2,12,0.000,-1.000
3,18,-1.000,-0.000
4,23,-0.259,0.966


Selected time model inputs: ['hour_sin', 'hour_cos']


In [ ]:
event_inputs_by_partition = {}
event_time_by_partition = {}
coverage_rows = []
weekday_names = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

for partition in PARTITIONS:
    path = PROCESSED_DIR / partition / 'events.csv'
    events = pd.read_csv(
        path, usecols=['event_key', 'timestamp', 'history_ids'],
        dtype='string', keep_default_na=False, encoding='utf-8',
    )
    event_inputs_by_partition[partition] = events
    event_time = derive_event_time_features(events)
    event_time_by_partition[partition] = event_time
    dates = event_time['timestamp'].dt.normalize().drop_duplicates()
    weekdays = sorted(event_time['day_of_week'].unique().tolist())
    coverage_rows.append({
        'partition': partition,
        'events': len(event_time),
        'earliest_timestamp': event_time['timestamp'].min(),
        'latest_timestamp': event_time['timestamp'].max(),
        'distinct_dates': len(dates),
        'observed_weekdays': ', '.join(weekday_names[d] for d in weekdays),
        'weekend_events': int(event_time['is_weekend'].sum()),
        'max_distinct_dates_for_one_weekday': int(dates.dt.dayofweek.value_counts().max()),
    })

if 'train' not in event_time_by_partition:
    raise ValueError('A training partition is needed to fit the category vocabulary.')
all_event_keys = pd.concat([f['event_key'] for f in event_time_by_partition.values()], ignore_index=True)
if not all_event_keys.is_unique:
    raise ValueError('Event keys are repeated across partitions.')

training_weekdays = set(event_time_by_partition['train']['day_of_week'])
time_coverage = pd.DataFrame(coverage_rows)
time_coverage['weekdays_absent_from_training'] = [
    ', '.join(weekday_names[d] for d in sorted(set(event_time_by_partition[p]['day_of_week']) - training_weekdays))
    for p in PARTITIONS
]
display(time_coverage)
if time_coverage.loc[time_coverage['partition'].eq('train'), 'max_distinct_dates_for_one_weekday'].iloc[0] < 2:
    warnings.warn('Training has no repeated weekdays. Treat weekly patterns as exploratory.', stacklevel=2)
if time_coverage['weekdays_absent_from_training'].ne('').any():
    print('Some held-out weekdays are absent from training; see coverage table.')
display(event_time_by_partition['train'].head())


,partition,events,earliest_timestamp,latest_timestamp,distinct_dates,observed_weekdays,weekend_events,max_distinct_dates_for_one_weekday,weekdays_absent_from_training
0,train,95071,2019-11-09 00:00:19,2019-11-12 23:59:25,4,"Monday, Tuesday, Saturday, Sunday",28618,1,
1,validation,61894,2019-11-13 00:01:00,2019-11-14 23:59:13,2,"Wednesday, Thursday",0,1,"Wednesday, Thursday"
2,test,73152,2019-11-15 00:00:01,2019-11-15 23:58:03,1,Friday,0,1,Friday


Some held-out weekdays are absent from training; see coverage table.


/usr/local/lib/python3.13/dist-packages/IPython/core/interactiveshell.py:3553: UserWarning: Training has no repeated weekdays. Treat weekly patterns as exploratory.
  exec(code_obj, self.user_global_ns, self.user_ns)


,event_key,timestamp,hour_of_day,fractional_hour,day_of_week,is_weekend,hour_sin,hour_cos,weekday_sin,weekday_cos
0,MINDsmall_train/behaviors.tsv:20112,2019-11-09 00:00:19,0,0.005278,5,1,0.001382,0.999999,-0.974928,-0.222521
1,MINDsmall_train/behaviors.tsv:13807,2019-11-09 00:01:13,0,0.020278,5,1,0.005309,0.999986,-0.974928,-0.222521
2,MINDsmall_train/behaviors.tsv:27660,2019-11-09 00:02:44,0,0.045556,5,1,0.011926,0.999929,-0.974928,-0.222521
3,MINDsmall_train/behaviors.tsv:152217,2019-11-09 00:02:50,0,0.047222,5,1,0.012362,0.999924,-0.974928,-0.222521
4,MINDsmall_train/behaviors.tsv:42166,2019-11-09 00:03:09,0,0.052500,5,1,0.013744,0.999906,-0.974928,-0.222521


## 7. Combine time with content and prior interests

All candidates in an impression share its timestamp. Adding only a common time bias cannot change their ordering. The scoring model needs a **candidate-dependent effect**, either learned by a nonlinear MLP that receives time and article/history vectors together, or supplied as explicit interaction features.

This notebook implements explicit, inspectable inputs:

1. One-hot candidate category, using a vocabulary fitted on **training news only**. Unseen/missing categories map to a reserved unknown category.
2. Each selected time feature multiplied by that category indicator. With weekday features enabled, this includes `is_weekend × entertainment`, for example. The model learns the weight and direction from clicks during later model training.
3. The share of known-category articles in this impression's supplied prior history matching the candidate category, plus a missing-history-category flag. Each selected time feature is also multiplied by this share.

The category-history share is a simple interest signal, not a time-specific user habit estimate. It supplements the richer user-history vector in your advanced models. Unknown categories do not count as interest matches. No history timestamps or future clicks are inferred. This stage never reads candidate click labels.

Profiles are normalised over history entries whose categories are known to the training vocabulary. Repeated history entries retain their supplied multiplicity. A history with no usable category gets zero shares and an explicit missing flag.


In [ ]:
article_categories_by_partition = {}


def load_article_categories(partition):
    frame = pd.read_csv(
        PROCESSED_DIR / partition / 'news.csv',
        usecols=['news_id', 'category', 'metadata_missing'],
        dtype='string', keep_default_na=False, encoding='utf-8',
    )
    if not frame['news_id'].is_unique:
        raise ValueError('Article-category lookup must have unique news_id values.')
    if not frame['metadata_missing'].isin(['True', 'False']).all():
        raise ValueError('Invalid metadata_missing flag in article-category lookup.')
    return frame


training_categories = load_article_categories('train')
known = (
    training_categories['metadata_missing'].eq('False')
    & training_categories['category'].str.strip().ne('')
    & training_categories['category'].ne('__UNKNOWN__')
)
category_to_id = {category: i + 1 for i, category in enumerate(
    sorted(training_categories.loc[known, 'category'].unique().tolist())
)}
CATEGORY_COUNT = len(category_to_id) + 1  # ID 0 is reserved for unknown.
category_mapping = pd.DataFrame([
    {'category_id': 0, 'category': '__UNKNOWN__'},
    *[{'category_id': code, 'category': name} for name, code in category_to_id.items()],
])
CATEGORY_COLUMNS = [f'candidate_category_{i:02d}' for i in range(CATEGORY_COUNT)]
HISTORY_SHARE_COLUMNS = [f'history_category_share_{i:02d}' for i in range(CATEGORY_COUNT)]

for partition in PARTITIONS:
    frame = training_categories.copy() if partition == 'train' else load_article_categories(partition)
    frame['category_id'] = frame['category'].map(category_to_id).fillna(0).astype('int32')
    frame.loc[frame['metadata_missing'].eq('True'), 'category_id'] = 0
    article_categories_by_partition[partition] = frame[['news_id', 'category', 'category_id']]
display(category_mapping)


def derive_history_category_profile(events, article_categories):
    lookup = dict(zip(article_categories['news_id'], article_categories['category_id']))
    values = np.zeros((len(events), CATEGORY_COUNT), dtype='float32')
    total_counts = np.zeros(len(events), dtype='int32')
    known_counts = np.zeros(len(events), dtype='int32')
    for row, (event_key, text) in enumerate(zip(events['event_key'], events['history_ids'])):
        try:
            history = json.loads(text)
        except (TypeError, json.JSONDecodeError) as exc:
            raise ValueError(f'{event_key}: invalid history_ids JSON') from exc
        if not isinstance(history, list) or not all(isinstance(n, str) for n in history):
            raise ValueError(f'{event_key}: history_ids must be a list of news IDs')
        total_counts[row] = len(history)
        # Missing metadata or an unseen category maps to 0 and contributes no match.
        codes = np.array([lookup.get(n, 0) for n in history], dtype='int32')
        codes = codes[codes > 0]
        known_counts[row] = len(codes)
        if len(codes):
            values[row] = np.bincount(codes, minlength=CATEGORY_COUNT) / len(codes)
    out = events[['event_key']].copy()
    out['history_article_count'] = total_counts
    out['history_known_category_count'] = known_counts
    out['history_category_missing'] = (known_counts == 0).astype('int8')
    shares = pd.DataFrame(values, columns=HISTORY_SHARE_COLUMNS, index=events.index)
    return pd.concat([out, shares], axis=1)


history_profiles_by_partition = {
    p: derive_history_category_profile(event_inputs_by_partition[p], article_categories_by_partition[p])
    for p in PARTITIONS
}
print('Category vocabulary fitted on training metadata only; 0 = unknown.')
display(history_profiles_by_partition['train'].head())


,category_id,category
0,0,__UNKNOWN__
1,1,autos
2,2,entertainment
3,3,finance
4,4,foodanddrink
5,5,health
6,6,kids
7,7,lifestyle
8,8,middleeast
9,9,movies


Category vocabulary fitted on training metadata only; 0 = unknown.


,event_key,history_article_count,history_known_category_count,history_category_missing,history_category_share_00,history_category_share_01,history_category_share_02,history_category_share_03,history_category_share_04,history_category_share_05,...,history_category_share_08,history_category_share_09,history_category_share_10,history_category_share_11,history_category_share_12,history_category_share_13,history_category_share_14,history_category_share_15,history_category_share_16,history_category_share_17
0,MINDsmall_train/behaviors.tsv:20112,15,15,0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.000000,0.000000,0.400000,0.0,0.466667,0.000000,0.066667,0.066667,0.00
1,MINDsmall_train/behaviors.tsv:13807,93,93,0,0.0,0.096774,0.064516,0.043011,0.021505,0.021505,...,0.0,0.021505,0.021505,0.397849,0.0,0.053763,0.096774,0.043011,0.000000,0.00
2,MINDsmall_train/behaviors.tsv:27660,7,7,0,0.0,0.142857,0.000000,0.000000,0.000000,0.000000,...,0.0,0.000000,0.142857,0.285714,0.0,0.285714,0.000000,0.000000,0.000000,0.00
3,MINDsmall_train/behaviors.tsv:152217,8,8,0,0.0,0.000000,0.250000,0.000000,0.000000,0.000000,...,0.0,0.000000,0.000000,0.500000,0.0,0.000000,0.125000,0.125000,0.000000,0.00
4,MINDsmall_train/behaviors.tsv:42166,4,4,0,0.0,0.250000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.000000,0.000000,0.500000,0.0,0.000000,0.000000,0.000000,0.000000,0.25


In [ ]:
def context_model_columns(time_columns):
    return [
        *time_columns, *CATEGORY_COLUMNS,
        *[f'{t}__x__{c}' for t in time_columns for c in CATEGORY_COLUMNS],
        'candidate_history_category_share', 'history_category_missing',
        *[f'{t}__x__history_category_share' for t in time_columns],
    ]


CONTEXT_MODEL_COLUMNS = context_model_columns(TIME_FEATURE_COLUMNS)


def build_candidate_context_features(candidate_batch, event_time, article_categories,
                                     history_profiles, time_columns=None):
    # Safe to call on minibatches; labels are neither required nor returned.
    time_columns = TIME_FEATURE_COLUMNS if time_columns is None else list(time_columns)
    if not time_columns or not set(time_columns).issubset(ALL_TIME_FEATURE_COLUMNS):
        raise ValueError('Select known, nonempty time feature columns.')
    if len(time_columns) != len(set(time_columns)):
        raise ValueError('Time feature selection contains duplicates.')
    keys = ['event_key', 'candidate_position', 'news_id']
    joined = candidate_batch[keys].merge(
        event_time[['event_key', *time_columns]],
        on='event_key', how='left', validate='many_to_one', sort=False,
    ).merge(
        article_categories[['news_id', 'category_id']],
        on='news_id', how='left', validate='many_to_one', sort=False,
    ).merge(
        history_profiles[['event_key', 'history_category_missing', *HISTORY_SHARE_COLUMNS]],
        on='event_key', how='left', validate='many_to_one', sort=False,
    )
    numeric = [*time_columns, 'category_id', 'history_category_missing', *HISTORY_SHARE_COLUMNS]
    if len(joined) != len(candidate_batch) or joined[numeric].isna().any().any():
        raise ValueError('Context join lost rows or coverage; use matching partition lookups.')
    codes = joined['category_id'].to_numpy(dtype='int32')
    indicators = np.eye(CATEGORY_COUNT, dtype='float32')[codes]
    shares = joined[HISTORY_SHARE_COLUMNS].to_numpy(dtype='float32')
    matching_interest = shares[np.arange(len(joined)), codes]
    values = {t: joined[t].to_numpy(dtype='float32') for t in time_columns}
    values.update({c: indicators[:, i] for i, c in enumerate(CATEGORY_COLUMNS)})
    for t in time_columns:
        for i, c in enumerate(CATEGORY_COLUMNS):
            values[f'{t}__x__{c}'] = values[t] * indicators[:, i]
    values['candidate_history_category_share'] = matching_interest
    values['history_category_missing'] = joined['history_category_missing'].to_numpy(dtype='float32')
    for t in time_columns:
        values[f'{t}__x__history_category_share'] = values[t] * matching_interest
    out = pd.concat([
        joined[keys + ['category_id']].reset_index(drop=True),
        pd.DataFrame(values, dtype='float32'),
    ], axis=1)
    if not np.isfinite(out[context_model_columns(time_columns)].to_numpy()).all():
        raise ValueError('Context model inputs contain non-finite values.')
    return out


preview_partition = 'train'
candidate_preview = pd.read_csv(
    PROCESSED_DIR / preview_partition / 'candidates.csv', nrows=8,
    usecols=['event_key', 'candidate_position', 'news_id'],
    dtype={'event_key': 'string', 'news_id': 'string', 'candidate_position': 'int32'},
    keep_default_na=False,
)
context_preview = build_candidate_context_features(
    candidate_preview, event_time_by_partition[preview_partition],
    article_categories_by_partition[preview_partition], history_profiles_by_partition[preview_partition],
)
print('Context input width for the chosen configuration:', len(CONTEXT_MODEL_COLUMNS))
display(context_preview[['event_key', 'news_id', 'category_id', *TIME_FEATURE_COLUMNS,
                         'candidate_history_category_share', 'history_category_missing']])


Context input width for the chosen configuration: 60


,event_key,news_id,category_id,hour_sin,hour_cos,candidate_history_category_share,history_category_missing
0,MINDsmall_train/behaviors.tsv:20112,N54300,2,0.001382,0.999999,0.000000,0.0
1,MINDsmall_train/behaviors.tsv:20112,N46057,11,0.001382,0.999999,0.400000,0.0
2,MINDsmall_train/behaviors.tsv:20112,N57005,11,0.001382,0.999999,0.400000,0.0
3,MINDsmall_train/behaviors.tsv:20112,N52154,7,0.001382,0.999999,0.000000,0.0
4,MINDsmall_train/behaviors.tsv:20112,N57099,11,0.001382,0.999999,0.400000,0.0
5,MINDsmall_train/behaviors.tsv:20112,N31002,13,0.001382,0.999999,0.466667,0.0
6,MINDsmall_train/behaviors.tsv:20112,N37108,11,0.001382,0.999999,0.400000,0.0
7,MINDsmall_train/behaviors.tsv:20112,N58075,11,0.001382,0.999999,0.400000,0.0


## 8. Export time context and reusable lookups

For every existing partition, save:

- `event_time_features.csv`: one row per `event_key`, including all derived time fields and the timestamp for audit.
- `article_categories.csv`: candidate/history category lookup by `news_id`.
- `event_history_category_profile.csv`: one row per `event_key`, with category shares from supplied prior history.
- The earlier `news_features.csv` retains the original three news features and missingness flags.

The root folder also contains `category_mapping.csv`, `time_coverage.csv`, the extended `feature_dictionary.csv`, and a manifest recording the selected model inputs and source checksums.

Use `build_candidate_context_features` on model minibatches to avoid a large repeated CSV. Optional full candidate export is implemented below and reads only candidate keys, in chunks. It creates no labels or recommendation scores.


In [ ]:
for partition in PARTITIONS:
    folder = OUTPUT_DIR / partition
    event_time_by_partition[partition].to_csv(
        folder / 'event_time_features.csv', index=False, encoding='utf-8',
        date_format='%Y-%m-%d %H:%M:%S',
    )
    article_categories_by_partition[partition].to_csv(folder / 'article_categories.csv', index=False)
    history_profiles_by_partition[partition].to_csv(folder / 'event_history_category_profile.csv', index=False)
category_mapping.to_csv(OUTPUT_DIR / 'category_mapping.csv', index=False)
time_coverage.to_csv(OUTPUT_DIR / 'time_coverage.csv', index=False)

extra_definitions = [
    ('hour_sin', 'sin(2*pi*fractional_hour/24)', 'float32', 'Daily cycle coordinate'),
    ('hour_cos', 'cos(2*pi*fractional_hour/24)', 'float32', 'Daily cycle coordinate'),
    ('weekday_sin', 'sin(2*pi*day_of_week/7); Monday=0', 'float32', 'Optional weekly coordinate'),
    ('weekday_cos', 'cos(2*pi*day_of_week/7); Monday=0', 'float32', 'Optional weekly coordinate'),
    ('is_weekend', '1 for recorded Saturday/Sunday, else 0', 'int8', 'Optional binary context'),
    ('hour_of_day', 'Integer hour 0..23; diagnostic only', 'int8', 'Hour'),
    ('fractional_hour', 'hour + minute/60 + second/3600; diagnostic only', 'float32', 'Hours'),
    ('day_of_week', 'Monday=0 through Sunday=6; diagnostic only', 'int8', 'Weekday index'),
    ('candidate_history_category_share', 'Known-category prior-history share matching the candidate', 'float32', 'Fraction'),
    ('history_category_missing', '1 if supplied history has no usable categories', 'int8', 'Binary flag'),
    ('<time>__x__candidate_category_<id>', 'Selected time feature times one-hot candidate category', 'float32', 'Interaction'),
    ('<time>__x__history_category_share', 'Selected time feature times candidate matching interest', 'float32', 'Interaction'),
]
extra_dictionary = pd.DataFrame([
    {'feature': name, 'source': 'Behaviors' if name in ALL_TIME_FEATURE_COLUMNS or name in
        ['hour_of_day', 'fractional_hour', 'day_of_week'] else 'Behaviors + News',
     'description': description, 'data_type': dtype, 'unit': unit, 'status': 'Derived'}
    for name, description, dtype, unit in extra_definitions
])
pd.concat([feature_dictionary, extra_dictionary], ignore_index=True).to_csv(
    OUTPUT_DIR / 'feature_dictionary.csv', index=False,
)

provenance.update({
    'source_events_sha256': {p: sha256_file(PROCESSED_DIR / p / 'events.csv') for p in PARTITIONS},
    'timestamp_timezone': 'Unspecified in source; preserved as recorded, timezone-naive',
    'time_feature_columns_available': ALL_TIME_FEATURE_COLUMNS,
    'time_feature_columns_selected': TIME_FEATURE_COLUMNS,
    'use_weekday_features': USE_WEEKDAY_FEATURES,
    'context_model_columns': CONTEXT_MODEL_COLUMNS,
    'category_vocabulary_fit': 'train/news.csv only, excluding missing metadata',
    'category_unknown_id': 0,
    'history_profile_source': 'Only this impression supplied history_ids; no future clicks',
    'weekly_pattern_status': 'Exploratory in the short observation window',
    'numpy_version': np.__version__,
})
(OUTPUT_DIR / 'manifest.json').write_text(json.dumps(provenance, indent=2) + '\n', encoding='utf-8')
print('Saved news and time features to:', OUTPUT_DIR)


Saved news and time features to: /content/drive/MyDrive/BT4222 Group Project/feature_engineering_zp/outputs/features/news_time_v3


In [ ]:
def export_candidate_context_features(partition, chunk_size=CANDIDATE_CHUNK_SIZE):
    if chunk_size < 1:
        raise ValueError('chunk_size must be positive.')
    target = OUTPUT_DIR / partition / 'candidate_context_features.csv'
    temporary = target.with_suffix('.csv.partial')
    rows_written = 0
    try:
        for batch in pd.read_csv(
            PROCESSED_DIR / partition / 'candidates.csv', chunksize=chunk_size,
            usecols=['event_key', 'candidate_position', 'news_id'],
            dtype={'event_key': 'string', 'news_id': 'string', 'candidate_position': 'int32'},
            keep_default_na=False,
        ):
            context = build_candidate_context_features(
                batch, event_time_by_partition[partition],
                article_categories_by_partition[partition], history_profiles_by_partition[partition],
            )
            context.to_csv(temporary, mode='w' if rows_written == 0 else 'a',
                           header=rows_written == 0, index=False)
            rows_written += len(context)
        if rows_written == 0:
            raise ValueError('No candidate rows to export.')
        temporary.replace(target)
    except Exception:
        temporary.unlink(missing_ok=True)
        raise
    print(f'{partition}: exported {rows_written:,} candidate contexts')
    return target


if EXPORT_CANDIDATE_CONTEXT:
    for partition in PARTITIONS:
        export_candidate_context_features(partition)
else:
    print('Compact tables exported. Candidate interactions are available via the batch builder.')


Compact tables exported. Candidate interactions are available via the batch builder.


## 9. Use the inputs in your advanced models

In this notebook, the in-memory lookups and `build_candidate_context_features` are ready to use. For a model minibatch:

```python
context = build_candidate_context_features(
    candidate_batch,
    event_time_by_partition[partition],
    article_categories_by_partition[partition],
    history_profiles_by_partition[partition],
)
X_context = context[CONTEXT_MODEL_COLUMNS].to_numpy(dtype='float32')
```

Reload the compact CSVs with string keys when running from a separate notebook; use the same training category mapping and column order. CSV does not preserve pandas dtypes. Keep the batch builder function with your model input code. You can also use the optional full candidate export.

For either advanced model, concatenate `X_context` with the candidate vector, the mean-pooled or candidate-attention history vector, and your other engineered features **before the MLP scoring head**. Article-only counts still join on `news_id`; impression context joins on `event_key`. Preserve `candidate_position` for row alignment, but do not pass IDs, timestamps or that position to the MLP as numeric inputs.

The explicit crosses permit candidate-specific time effects even with a linear scoring head. A nonlinear MLP can also learn interactions directly from time/category/history inputs; compare raw concatenation against explicit crosses if useful. Neither the encoding nor the cross products guarantee better ranking.

For evaluation, compare the same architecture with: **no time → hour context → optional hour + weekday/weekend context**. For a fair no-time ablation, retain category and history-share inputs and remove both the time columns and **all** columns containing `__x__`. Use the same complete candidate lists, validation protocol and seeds; select using validation nDCG@10 and reserve test labels for final evaluation. Do not interpret absent validation weekends as evidence that weekend behaviour was validated.

## Source and verification scope

The current schema and split were checked against [`src/preprocessing.py`](https://github.com/JariusF10-8/bt4222/blob/preprocessing/src/preprocessing.py), [`src/data_loading.py`](https://github.com/JariusF10-8/bt4222/blob/preprocessing/src/data_loading.py), and the [data dictionary](https://github.com/JariusF10-8/bt4222/blob/preprocessing/data/data_dictionary.md) on 25 October 2026.

Microsoft documents the [MIND impression timestamp and supplied prior history](https://learn.microsoft.com/en-us/azure/open-datasets/dataset-microsoft-news#behaviorstsv). The [scikit-learn time-feature example](https://scikit-learn.org/stable/auto_examples/applications/plot_cyclical_feature_engineering.html#trigonometric-features) explains cyclic sine/cosine encoding and its limits.

All code cells were checked on small fixtures generated by the repository's canonical preprocessing helper, including midnight/noon, seconds, weekday/weekend boundaries, train-only category mapping, empty histories, unknown categories, candidate row alignment and chunked export. The full group dataset is not bundled: run on your processed version to obtain actual tables and coverage. Outputs are cleared for a clean run.
